In [ ]:
from urllib.request import urlretrieve

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

In [31]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=256)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [22]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt"
urlretrieve(URL, "companyPolicies.txt")
# print(open("companyPolicies.txt").read()[:300])

documents = TextLoader("companyPolicies.txt").load()
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = splitter.split_documents(documents)
print(len(texts))
print([len(t.page_content) for t in texts])


recur_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = recur_splitter.split_documents(documents=documents)
print(len(chunks), max(len(c.page_content) for c in chunks))
print([len(t.page_content) for t in chunks])

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16
[18, 1624, 21, 1885, 28, 1903, 22, 1729, 17, 1678, 26, 2032, 769, 1894, 36, 1947]
50 499
[18, 498, 274, 441, 408, 21, 262, 495, 488, 311, 325, 28, 300, 443, 478, 378, 300, 22, 454, 402, 412, 458, 17, 498, 486, 421, 270, 26, 496, 409, 437, 377, 309, 27, 495, 244, 44, 323, 326, 290, 296, 233, 421, 36, 243, 260, 272, 229, 499, 439]


In [ ]:
docsearch = InMemoryVectorStore.from_documents(chunks, embeddings)
print("docs ingested")

docs ingested


In [38]:
hits = docsearch.similarity_search_with_score("can i smoke in company vehicles", k=6)
for doc, score in hits:
    print(round(score, 2), doc.page_content[:80].replace("\n", " "))

0.71 No Smoking in Company Vehicles: Smoking is not permitted in company vehicles, wh
0.56 Smoking Restrictions: Smoking inside company buildings, offices, meeting rooms, 
0.5 5.	Smoking Policy
0.49 Policy Purpose: The Smoking Policy has been established to provide clear guidanc
0.46 Alcohol Consumption: The consumption of alcoholic beverages is not allowed durin
0.39 Review of Policy: This policy will be reviewed periodically to ensure its alignm


In [42]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})


def format_docs(docs):
    return "\n \n".join(d.page_content for d in docs)


print(format_docs(retriever.invoke("What is the mobile phone policy?")))

4.	Mobile Phone Policy
 
The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.
Acceptable Use: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.
 
Consequences: Non-compliance with this policy may lead to disciplinary actions, including the potential loss of mobile phone privileges.
The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.


In [70]:
prompt = ChatPromptTemplate.from_template(
    "Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}"
)
strict_prompt = ChatPromptTemplate.from_template(
    "Use only the context below to answer the question at the end. "
    'If the answer is not in the context, say "I don\'t know". '
    "Do not make up an answer.\n\nContext:\n{context}\n\nQuestion: {question}"
)
parser = StrOutputParser()

qa = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)
qa_strict = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | strict_prompt
    | llm
    | parser
)

print(qa_strict.invoke("What is the mobile policy"))
print(qa_strict.invoke("What is the phone policy"))
print(qa_strict.invoke("What is the mobile phone policy"))
print(qa_strict.invoke("What is the sexual policy"))
print(qa_strict.invoke("What is the fine dining policy"))


I don't know.
The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices in the organization. It is intended to ensure that employees use mobile phones primarily for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations. Non-compliance with the policy may result in disciplinary actions, including the potential loss of mobile phone privileges. The policy aims to promote responsible and secure use of mobile devices in line with legal and ethical standards, and employees are expected to understand and comply with these guidelines. Regular reviews of the policy ensure its alignment with evolving technology and security best practices.
The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. Its purpose is to ensure that employees utilize mobile phones in a manner consistent with c

In [ ]:
history = []

parser = StrOutputParser()

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user's last question so it makes sense on its own, using the chat history. Return only the question.",
        ),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

rewrite = rewrite_prompt | llm | parser
chain = strict_prompt | llm | parser


def ask(question):
    standalone = (
        rewrite.invoke({"history": history, "question": question})
        if history
        else question
    )
    context = format_docs(retriever.invoke(standalone))
    answer = chain.invoke({"context": context, "question": standalone})
    history.extend([HumanMessage(question), AIMessage(answer)])
    return answer


history.clear()
print(ask("What is the mobile phone policy?"))
print("----")
print(ask("List the points in it."))
print("----")
print(ask("What is the aim of it?"))


The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. Its purpose is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance. Mobile devices are primarily intended for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations. Non-compliance with the policy may lead to disciplinary actions, including the potential loss of mobile phone privileges. The policy aims to promote the responsible and secure use of mobile devices in line with legal and ethical standards, and every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.
----
The key points of the Mobile Phone Policy are:

1. It promotes the responsible and secure use of mobile devices in alignment with legal and

In [ ]:
def chat():
    history.clear()

    while True:
        question = input("Question: ")
        if question.lower() in ["quit", "exit", "bye"]:
            print("Goodbye!")
            break
        print("Answer: ", ask(question))


In [77]:
chat()

KeyboardInterrupt: 